# 01 — All boxes, OCR and labels
Reuse our development `magi_raw.json` files. Missing raw files are created below. No detected boxes are filtered.


In [1]:
# Cell purpose: Load preparation settings and check whether the canonical sequence cache exists.
# Inputs: config.py and existing cache files.
# Outputs: Paths/imports needed below and the reuse status of sequences.json.

from pathlib import Path
import json
from config import CONFIG, ROOT, fingerprint, save_json, require_cache, run_stage, model_files

# Edit config.py once. Restart the kernel after changing it.
# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
CACHE.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print('Existing caches:', [p.name for p in CACHE.iterdir() if p.is_file()])

import re
import hashlib
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from PIL import Image

SEQUENCES = CACHE / 'sequences.json'
print('Reuse:', SEQUENCES if SEQUENCES.exists() else 'create below')

Existing caches: ['discovery.json']
Reuse: create below


In [2]:
# Cell purpose: Define OCR normalization, confident GT matching, label mapping and missing-cache creation.
# Inputs: OCR strings, GT page entries and original page images.
# Outputs: Reusable functions; Magi is loaded only when create_raw is actually called.

raw_model = None

# Normalize case, Unicode and punctuation before comparing OCR with GT text.
def normalize(text):
    text = unicodedata.normalize('NFKC', text or '').casefold()
    words = ' '.join(''.join(c if c.isalnum() else ' ' for c in text).split())
    return words or ' '.join(text.split())

# Require both a high similarity score and a clear margin over the second GT candidate.
def match_gt(ocr, lines):
    text = normalize(ocr)
    if not text:
        return None
    scores = sorted([(100 * SequenceMatcher(None, text, normalize(x['text']), autojunk=False).ratio(), i)
                     for i, x in enumerate(lines) if normalize(x['text'])], reverse=True)
    if not scores or scores[0][0] < CONFIG['match_threshold']:
        return None
    if len(scores) > 1 and scores[0][0] - scores[1][0] < CONFIG['match_margin']:
        return None
    return scores[0][1]

# Map character speakers to SPOKEN=0, narration to NARRATED=1, and other speakers to ignore=-100.
def label_for(speaker):
    if speaker == 'NARRATION':
        return 1
    if re.fullmatch(r'character[1-9][0-9]*', speaker or ''):
        return 0
    return -100

# Fallback only: run frozen Magi detection/OCR when the sequence has no reusable raw cache.
def create_raw(page_paths):
    import numpy as np
    import torch
    from transformers import AutoModel
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    global raw_model
    if raw_model is None:
        raw_model = AutoModel.from_pretrained(CONFIG['magi_model'], trust_remote_code=True, local_files_only=True).to(device).eval()
    model = raw_model
    model.requires_grad_(False)
    pages = []
    # Disable autograd for frozen feature extraction or evaluation to reduce memory use.
    with torch.inference_mode():
        for path in page_paths:
            with Image.open(path) as image:
                image = image.convert('L').convert('RGB') if CONFIG['grayscale'] else image.convert('RGB')
                array = np.asarray(image)
            page = model.predict_detections_and_associations([array],
                text_detection_threshold=CONFIG['text_threshold'],
                tail_detection_threshold=CONFIG['tail_detection_threshold'],
                text_tail_matching_threshold=CONFIG['tail_match_threshold'])[0]
            page['ocr'] = model.predict_ocr([array], [page['texts']])[0]
            pages.append(page)
    return json_ready({'data': pages})

# Convert arrays/tensors recursively to plain JSON-compatible values before saving raw data.
def json_ready(value):
    if hasattr(value, 'detach'):
        return value.detach().cpu().tolist()
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {k: json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value

In [3]:
# Cell purpose: Reuse or build complete three-page sequences while retaining every detected text box.
# Inputs: GT JSONL, three ordered images per sequence and existing raw Magi/OCR caches.
# Outputs: cache/sequences.json, including labeled and ignored boxes in their original order.

# Reusing sequences.json preserves the previous box order and matched labels.
if SEQUENCES.exists():
    sequences = json.loads(SEQUENCES.read_text(encoding='utf-8'))
else:
    labels = [json.loads(line) for line in CONFIG['labels_jsonl'].read_text(encoding='utf-8').splitlines() if line.strip()]
    assert len({x['sequence_id'] for x in labels}) == len(labels)
    sequences = []
    for row in labels:
        sid = row['sequence_id']
        assert len(row['pages']) == 3
        paths = sorted((CONFIG['images_root'] / sid).glob('*.png'))
        assert len(paths) == 3, f'{sid}: expected three PNG pages; adjust image discovery for another format'
        # Search existing project caches first; only new fallback raw files go into this experiment cache.
        candidates = [p / sid / 'magi_raw.json' for p in CONFIG['raw_cache_roots']]
        candidates.append(CACHE / 'raw' / sid / 'magi_raw.json')
        raw_path = next((p for p in candidates if p.exists()), candidates[-1])
        if raw_path.exists():
            saved = json.loads(raw_path.read_text(encoding='utf-8'))
        else:
            saved = create_raw(paths)
            saved['settings'] = {'pages': [{'name': p.name, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in paths]}
            save_json(raw_path, saved)
        assert len(saved['data']) == 3, sid
        # Verify the cached payload checksum before assigning labels or using its boxes.
        if 'data_hash' in saved:
            assert fingerprint(saved['data']) == saved['data_hash'], f'Corrupt cache: {raw_path}'
        page_settings = saved.get('settings', saved.get('signature', {})).get('pages', [])
        boxes, page_info = [], []
        for page_idx, (path, page, lines) in enumerate(zip(paths, saved['data'], row['pages'])):
            # Check image identity so detections from a different page cannot be joined accidentally.
            sha = hashlib.sha256(path.read_bytes()).hexdigest()
            if page_settings:
                assert page_settings[page_idx]['name'] == path.name
                assert page_settings[page_idx]['sha256'] == sha, f'Image/cache mismatch: {path}'
            with Image.open(path) as image:
                width, height = image.size
            page_info.append({'path': str(path), 'sha256': sha, 'width': width, 'height': height})
            bboxes, ocr = page['texts'], page['ocr']
            assert len(bboxes) == len(ocr)
            # Match within this page only; multiple boxes claiming the same GT entry stay ignored.
            matches = [match_gt(text, lines) for text in ocr]
            counts = Counter(m for m in matches if m is not None)
            for text_idx, (bbox, text, match) in enumerate(zip(bboxes, ocr, matches)):
                gt = lines[match] if match is not None and counts[match] == 1 else None
                # Append every real box, including unmatched/UNKNOWN boxes that remain useful context.
                boxes.append({'box_id': f'PG{page_idx+1}:T{text_idx+1}', 'page_index': page_idx,
                    'text_index': text_idx, 'ocr': text, 'bbox': bbox,
                    'gt_speaker': gt['speaker'] if gt else None,
                    'label': label_for(gt['speaker']) if gt else -100})
        assert boxes, f'{sid} has no detected text boxes'
        sequences.append({'sequence_id': sid, 'pages': page_info, 'boxes': boxes, 'raw_cache': str(raw_path)})
    save_json(SEQUENCES, sequences)
print('Sequences:', len(sequences), '| Labels:', Counter(b['label'] for s in sequences for b in s['boxes']))
if raw_model is not None:
    del raw_model
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Sequences: 80 | Labels: Counter({0: 1427, -100: 693, 1: 63})


In [ ]:
# Cell purpose: Optional scratch cell; no operation is currently defined.
# Inputs: None.
# Outputs: None; this cell contains comments only.

